<a href="https://colab.research.google.com/github/splakplutoy/tugas-sisdas-cv/blob/main/tugas_sisdas_cv_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub -q

import os
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
IMG_SIZE = (300, 300)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

kagglehub.login()

print("\nMengunduh dataset...")
dataset_path = Path(kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product"))
print(f"Data berhasil diunduh dan siap digunakan di folder: {dataset_path}")

base_dir = dataset_path / 'casting_data' / 'casting_data'
if not base_dir.exists():
    base_dir = dataset_path / 'casting_data'

train_dir = base_dir / 'train'
test_dir = base_dir / 'test'

print("\nFolder dataset:")
print("Train:", train_dir)
print("Test :", test_dir)




In [ ]:
# Eksplorasi singkat dataset dan distribusi kelas
class_dirs = {
    'Train Defective': train_dir / 'def_front',
    'Train OK': train_dir / 'ok_front',
    'Test Defective': test_dir / 'def_front',
    'Test OK': test_dir / 'ok_front',
}

print("Distribusi dataset:")
for name, folder in class_dirs.items():
    count = len([p for p in folder.iterdir() if p.is_file()])
    print(f"{name:15}: {count}")

rng = random.Random(SEED)
ok_img_path = rng.choice(list((train_dir / 'ok_front').glob('*')))
def_img_path = rng.choice(list((train_dir / 'def_front').glob('*')))

fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].imshow(mpimg.imread(ok_img_path), cmap='gray')
ax[0].set_title(f'OK (Barang Bagus)\n{ok_img_path.name}')
ax[0].axis('off')

ax[1].imshow(mpimg.imread(def_img_path), cmap='gray')
ax[1].set_title(f'Defective (Barang Cacat)\n{def_img_path.name}')
ax[1].axis('off')

plt.tight_layout()
plt.show()




In [ ]:
# Data training dan validation dipisah dari folder train.
# Folder test hanya dipakai nanti untuk evaluasi final.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

print("Memuat Data Training:")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=SEED
)

print("\nMemuat Data Validation:")
val_generator = val_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=SEED,
    shuffle=False
)

print("\nIndeks Kelas:", train_generator.class_indices)




In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D
from tensorflow.keras.models import Sequential

model = Sequential([
    Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1)),

    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Ringkasan Arsitektur Model:")
model.summary()

callbacks = [
    ModelCheckpoint(
        'best_model_kustom_qc.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("\nMemulai proses training. Silakan tunggu...")
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=callbacks
)

model.save('model_kustom_qc.keras')
print("\nTraining selesai!")
print("Bobot model terbaik tersimpan sebagai: best_model_kustom_qc.weights.h5")
print("Model akhir tersimpan sebagai   : model_kustom_qc.keras")




In [ ]:
# Visualisasi kurva training untuk membaca overfitting/underfitting
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history['accuracy'], label='Train Accuracy')
ax[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
ax[0].set_title('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='Train Loss')
ax[1].plot(history.history['val_loss'], label='Validation Loss')
ax[1].set_title('Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()


